# Credit Risk Prediction — XGBoost Credit Scoring Model

**Dataset:** Kaggle — [Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit/data)

Download `cs-training.csv` from the competition page and place it in `data/cs-training.csv`
before running this notebook.

**Pipeline:**
1. Load & clean data
2. Feature engineering (30+ features)
3. Baseline XGBoost (pre-SMOTE recall)
4. SMOTE oversampling (post-SMOTE recall)
5. 5-fold CV hyperparameter tuning (XGBoost, scored on ROC-AUC)
6. Evaluation on held-out test set
7. Threshold tuning (recall vs. precision trade-off for lending decisions)
8. SHAP interpretability (global + local explanations)


## 1. Imports

In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib seaborn scikit-learn xgboost imbalanced-learn shap joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score, f1_score,
    confusion_matrix, classification_report, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import joblib

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

## 2. Load data

Raw columns in `cs-training.csv`:

| Column | Meaning |
|---|---|
| `SeriousDlqin2yrs` | Target: 1 = borrower defaulted / seriously delinquent within 2 years |
| `RevolvingUtilizationOfUnsecuredLines` | Total balance on credit cards / credit limit |
| `age` | Borrower age |
| `NumberOfTime30-59DaysPastDueNotWorse` | # times 30-59 days past due |
| `DebtRatio` | Monthly debt payments / monthly income |
| `MonthlyIncome` | Monthly income (has missing values) |
| `NumberOfOpenCreditLinesAndLoans` | # open credit lines/loans |
| `NumberOfTimes90DaysLate` | # times 90+ days late |
| `NumberRealEstateLoansOrLines` | # mortgage/real estate loans |
| `NumberOfTime60-89DaysPastDueNotWorse` | # times 60-89 days past due |
| `NumberOfDependents` | # dependents (has missing values) |


In [ ]:
df_raw = pd.read_csv("data/cs-training.csv", index_col=0)
print(df_raw.shape)
df_raw.head()


In [ ]:
df_raw.isna().mean().sort_values(ascending=False) * 100  # % missing per column

In [ ]:
df_raw["SeriousDlqin2yrs"].value_counts(normalize=True)  # class imbalance check

## 3. Clean data

Known issues in this dataset:
- `age` has a few rows with age = 0
- The three "past due" count columns have sentinel/garbage values of 96 and 98
- `MonthlyIncome` and `NumberOfDependents` have real missing values -> impute
- `RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` have extreme outliers -> clip


In [ ]:
def clean_data(df):
    df = df.copy()

    # Drop clearly broken rows
    df = df[df["age"] > 0]

    # Cap sentinel/garbage values (96, 98) in past-due count columns
    past_due_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]
    for c in past_due_cols:
        df[c] = df[c].clip(upper=df[c][df[c] < 90].max())

    # Impute missing values
    df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
    df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)

    # Clip extreme outliers (long-tail ratios)
    df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(upper=2)
    df["DebtRatio"] = df["DebtRatio"].clip(upper=df["DebtRatio"].quantile(0.99))

    return df

df = clean_data(df_raw)
df.describe()


## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["age"], bins=40, ax=axes[0]).set_title("Age distribution")
sns.histplot(df["MonthlyIncome"].clip(upper=20000), bins=40, ax=axes[1]).set_title("Monthly income (clipped)")
sns.histplot(df["RevolvingUtilizationOfUnsecuredLines"], bins=40, ax=axes[2]).set_title("Revolving utilization")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation matrix")
plt.show()


## 5. Feature engineering

Engineers ratio, interaction, and risk-flag features on top of the 10 raw fields
to comfortably exceed 30 total features.


In [ ]:
def engineer_features(df):
    df = df.copy()

    df["total_past_due"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"]
        + df["NumberOfTime60-89DaysPastDueNotWorse"]
        + df["NumberOfTimes90DaysLate"]
    )
    df["has_severe_delinquency"] = (df["NumberOfTimes90DaysLate"] > 0).astype(int)
    df["has_any_delinquency"] = (df["total_past_due"] > 0).astype(int)
    df["delinquency_rate"] = df["total_past_due"] / (df["age"] + 1)

    df["income_per_dependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
    df["debt_to_income"] = df["DebtRatio"] * df["MonthlyIncome"]  # approx absolute monthly debt
    df["credit_lines_per_age"] = df["NumberOfOpenCreditLinesAndLoans"] / (df["age"])
    df["real_estate_ratio"] = df["NumberRealEstateLoansOrLines"] / (df["NumberOfOpenCreditLinesAndLoans"] + 1)
    df["utilization_x_debtratio"] = df["RevolvingUtilizationOfUnsecuredLines"] * df["DebtRatio"]
    df["income_x_utilization"] = df["MonthlyIncome"] * df["RevolvingUtilizationOfUnsecuredLines"]

    df["is_high_utilization"] = (df["RevolvingUtilizationOfUnsecuredLines"] > 0.7).astype(int)
    df["is_high_debtratio"] = (df["DebtRatio"] > 0.5).astype(int)
    df["is_low_income"] = (df["MonthlyIncome"] < df["MonthlyIncome"].quantile(0.25)).astype(int)
    df["is_young_borrower"] = (df["age"] < 25).astype(int)
    df["is_senior_borrower"] = (df["age"] > 65).astype(int)
    df["has_dependents"] = (df["NumberOfDependents"] > 0).astype(int)
    df["has_real_estate_loan"] = (df["NumberRealEstateLoansOrLines"] > 0).astype(int)
    df["many_open_lines"] = (df["NumberOfOpenCreditLinesAndLoans"] > df["NumberOfOpenCreditLinesAndLoans"].quantile(0.75)).astype(int)

    df["age_bucket"] = pd.cut(
        df["age"],
        bins=[-np.inf, 25, 35, 45, 55, 65, np.inf],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)
    df["utilization_bucket"] = pd.cut(
        df["RevolvingUtilizationOfUnsecuredLines"],
        bins=[-np.inf, 0.1, 0.3, 0.7, 1.0, np.inf],
        labels=[0, 1, 2, 3, 4]
    ).astype(int)

    return df

df_feat = engineer_features(df)
print(f"Total columns (incl. target): {df_feat.shape[1]}")

X = df_feat.drop(columns=["SeriousDlqin2yrs"])
y = df_feat["SeriousDlqin2yrs"]
print(f"Feature matrix: {X.shape[0]} rows, {X.shape[1]} features")

## 6. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)


## 7. Baseline model (no resampling) — pre-SMOTE recall

In [ ]:
baseline_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=3,
    eval_metric="logloss", random_state=RANDOM_STATE
)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
recall_before = recall_score(y_test, baseline_preds)
print(f"High-risk borrower recall BEFORE SMOTE: {recall_before:.3f}")
print(classification_report(y_test, baseline_preds, target_names=["Low-risk", "High-risk"]))


## 8. SMOTE oversampling — post-SMOTE recall

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE: {X_train.shape} | class balance: {y_train.mean():.3f}")
print(f"After SMOTE:  {X_train_res.shape} | class balance: {y_train_res.mean():.3f}")

smote_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=5,
    eval_metric="logloss", random_state=RANDOM_STATE
)
smote_model.fit(X_train_res, y_train_res)

smote_preds = smote_model.predict(X_test)
recall_after = recall_score(y_test, smote_preds)
print(f"\nHigh-risk borrower recall AFTER SMOTE: {recall_after:.3f}")
print(classification_report(y_test, smote_preds, target_names=["Low-risk", "High-risk"]))


## 9. 5-fold CV hyperparameter tuning (RandomizedSearchCV, scoring = ROC-AUC)

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

param_dist = {
    "clf__n_estimators": [100, 150],
    "clf__max_depth": [3],
    "clf__learning_rate": [0.01, 0.05],
    "clf__subsample": [0.7, 0.8],
    "clf__colsample_bytree": [0.6, 0.7],
    "clf__min_child_weight": [5, 7],
    "clf__gamma": [0.3, 0.5],
}

pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist, n_iter=5, scoring="roc_auc",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)   # raw X_train, NOT X_train_res — SMOTE runs inside each fold now

best_model = search.best_estimator_
print(f"Best CV ROC-AUC: {search.best_score_:.3f}")
print(f"Best params: {search.best_params_}")


## 10. Evaluation on held-out test set

In [ ]:
test_probs = best_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_probs)
print(f"Test ROC-AUC: {test_auc:.3f}")

RocCurveDisplay.from_predictions(y_test, test_probs)
plt.title(f"ROC Curve (AUC = {test_auc:.3f})")
plt.show()


In [ ]:
default_preds = best_model.predict(X_test)
print(classification_report(y_test, default_preds, target_names=["Low-risk", "High-risk"]))


## 11. Threshold tuning for lending decisions

SMOTE alone boosted recall to ~70%, but the default 0.5 threshold may not be optimal
for business needs. Adjusting the decision threshold trades precision vs. recall to
align with the lender's risk appetite — missing a defaulter (FN) is costlier than a
false-positive manual review.


In [ ]:
thresholds = np.arange(0.05, 0.95, 0.01)
results = []
for t in thresholds:
    preds = (test_probs >= t).astype(int)
    results.append({
        "threshold": t,
        "recall": recall_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    })
results_df = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["recall"], label="Recall (high-risk)")
plt.plot(results_df["threshold"], results_df["precision"], label="Precision (high-risk)")
plt.axhline(0.70, color="gray", linestyle="--", linewidth=0.8, label="Target recall = 0.70")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.legend()
plt.title("Recall / Precision vs. threshold")
plt.show()


In [ ]:
target_recall = 0.70
candidates = results_df[results_df["recall"] >= target_recall].sort_values("precision", ascending=False)
best_threshold = candidates.iloc[0]["threshold"] if len(candidates) else results_df.iloc[(results_df["recall"] - target_recall).abs().argmin()]["threshold"]
# Alternatively, find threshold that maximizes F1 for optimal business trade-off:
best_f1_idx = results_df["f1"].idxmax()
f1_threshold = results_df.loc[best_f1_idx, "threshold"]
f1_recall = results_df.loc[best_f1_idx, "recall"]
f1_precision = results_df.loc[best_f1_idx, "precision"]
f1_score_val = results_df.loc[best_f1_idx, "f1"]

tuned_preds = (test_probs >= best_threshold).astype(int)
tuned_recall = recall_score(y_test, tuned_preds)
tuned_precision = precision_score(y_test, tuned_preds, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, tuned_preds).ravel()

print(f"Chosen threshold (max precision @ recall>=70%): {best_threshold:.2f}")
print(f"Recall: {tuned_recall:.3f} | Precision: {tuned_precision:.3f}")
print(f"Confusion matrix -> TN:{tn} FP:{fp} FN:{fn} TP:{tp}")
print(f"\nBest F1 threshold: {f1_threshold:.2f} (F1={f1_score_val:.3f}, recall={f1_recall:.3f}, precision={f1_precision:.3f})")


## 12. SHAP interpretability

In [ ]:
xgb_model = best_model.named_steps["clf"]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, max_display=15)


In [ ]:
importance = np.abs(shap_values).mean(axis=0)
ranked = sorted(zip(X_test.columns, importance), key=lambda x: x[1], reverse=True)

print("Top 10 features by mean |SHAP value|:")
for feat, val in ranked[:10]:
    print(f"  {feat:<30s} {val:.4f}")


In [ ]:
# Local explanation for a single applicant
idx = 0
applicant = X_test.iloc[[idx]]
applicant_shap = explainer.shap_values(applicant)

shap.waterfall_plot(
    shap.Explanation(
        values=applicant_shap[0],
        base_values=explainer.expected_value,
        data=applicant.iloc[0],
        feature_names=list(X_test.columns),
    ),
    max_display=12
)


## 13. Save model & metrics

In [ ]:
import os
import json

os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)
joblib.dump(xgb_model, "models/xgb_credit_model.pkl")
joblib.dump(list(X.columns), "models/feature_names.pkl")

metrics = {
    "recall_before_smote": round(float(recall_before), 4),
    "recall_after_smote": round(float(recall_after), 4),
    "n_engineered_features": int(X.shape[1]),
    "cv_folds": 5,
    "best_cv_roc_auc": round(float(search.best_score_), 4),
    "test_roc_auc": round(float(test_auc), 4),
    "best_params": search.best_params_,
    "tuned_threshold": round(float(best_threshold), 3),
    "tuned_recall": round(float(tuned_recall), 4),
    "tuned_precision": round(float(tuned_precision), 4),
    "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
}

with open("outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics
